In [12]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime, timedelta
import glob

In [15]:
# Define the directory and file pattern
file_directory = r"C:\Users\gctic.gsit05\Seguro Social del Salud\GCTIC GSIT DEV - Repositorio\SAD\16. Monitoreo Camas"
file_pattern = os.path.join(file_directory, "MonitoreoCamasEsSalud-*.xlsx")

# List all files matching the pattern
file_list_temp = glob.glob(file_pattern)

# Get today's date and the dates of the 4 previous days
today = datetime.now()
recent_dates = [(today - timedelta(days=i)).strftime('%Y%m%d') for i in range(5)]
recent_dates_timestamps = [(today - timedelta(days=i)).strftime('%Y-%m-%d') for i in range(5)]

# Filter files by the recent dates
file_list = [f for f in file_list_temp if os.path.basename(f).split('-')[1].split('.')[0] in recent_dates]

In [6]:
# Database connection string
DB_USER = 'postgres'
DB_PASSWORD = 'AKindOfMagic'
DB_NAME = "dw_essalud"
DB_PORT = "5432"
DB_HOST = '10.0.1.228'
cadena2 = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)

In [ ]:
# DATAFRAME DE CAMAS

def process_file(file_path):
    # Read the Excel file into a DataFrame
    df = pd.read_excel(file_path, sheet_name='IPRESS')
    df_2 = df.copy()
    
    # Get column names as a list
    columns_list = df.columns.tolist()

    # Add a new row at the top with the column names as values
    df.loc[-1] = columns_list

    # Reset index after inserting the new row
    df.index = df.index + 1
    df.sort_index(inplace=True)

    # Get the value from the first column and row
    value = df.iloc[0, 0]

    # Drop the first two rows
    df = df.drop(df.index[:2])

    # Reset the index
    df = df.reset_index(drop=True)

    # Keep the first 3 rows intact
    df_top = df.iloc[:4]

    # Drop rows starting from row 4, where the second column contains "SUB-TOTAL" or NaN
    df_bottom = df.iloc[4:].drop(df.iloc[4:].index[(df.iloc[4:][df.columns[1]].str.contains("TOTAL", na=False)) | (df.iloc[4:][df.columns[1]].str.contains("SUB-TOTAL", na=False)) | (df.iloc[4:][df.columns[1]].isna())])

    # Concatenate the two DataFrames
    df = pd.concat([df_top, df_bottom], ignore_index=True)

    df.iloc[0] = df.iloc[0].fillna(df.iloc[0].ffill())
    df.iloc[1] = df.iloc[1].fillna(df.iloc[1].ffill())
    df.iloc[2] = df.iloc[2].fillna(df.iloc[2].ffill())

    df.iloc[:4, :4] = df.iloc[:4, :4].ffill()

    # Drop columns where row 0 has either "TOTAL CAMAS IPRESS" or "RESUMEN"
    df = df.loc[:, ~df.iloc[0].isin(['TOTAL CAMAS IPRESS', 'RESUMEN'])]

    # Extract column names from the third row (assuming row 4 is index 3)
    new_columns = list(df.iloc[3])

    # Set new columns for the DataFrame
    df.columns = new_columns

    # Drop the third row (index 3)
    df = df.drop(3)

    # Reset the index
    df.reset_index(drop=True, inplace=True)

    # Get the first four columns
    base_columns = df.iloc[:, :4]

    # Find the number of 'Total' and 'Ocup' column pairs
    num_pairs = (df.shape[1] - 4) // 2

    # Create a list to store mini-dataframes
    mini_dfs = []

    # Loop through each pair and create a mini-dataframe
    for i in range(num_pairs):
        total_col = 4 + 2 * i
        ocup_col = total_col + 1
        mini_df = pd.concat([base_columns, df.iloc[:, [total_col, ocup_col]]], axis=1)
        
        # Extract the first 3 rows below 'Total' and 'Ocup' columns
        transposed_values = mini_df.iloc[:3, 4:].T
        transposed_values.columns = ['TIP_CAMA_1', 'TIP_CAMA_2', 'TIP_CAMA_3']
        
        # Reset index of the transposed dataframe to align with base columns
        transposed_values = transposed_values.reset_index(drop=True)
        
        # Drop duplicate rows in the transposed dataframe
        transposed_values = transposed_values.drop_duplicates().reset_index(drop=True)
        
        # Create a new dataframe with the base columns and transposed values
        new_mini_df = pd.concat([base_columns.iloc[0:1].reset_index(drop=True), transposed_values], axis=1)
        
        # Append the remaining rows of 'Total' and 'Ocup' columns
        remaining_df = pd.concat([base_columns.iloc[3:].reset_index(drop=True), mini_df.iloc[3:, 4:].reset_index(drop=True)], axis=1)
        
        # Combine the new mini dataframe with remaining values
        final_mini_df = pd.concat([new_mini_df, remaining_df], ignore_index=True)
        
        # Fill down NA values in 'Value1', 'Value2', 'Value3' columns
        final_mini_df[['TIP_CAMA_1', 'TIP_CAMA_2', 'TIP_CAMA_3']] = final_mini_df[['TIP_CAMA_1', 'TIP_CAMA_2', 'TIP_CAMA_3']].ffill()
        
        # Drop the 0th row of the final_mini_df
        final_mini_df = final_mini_df.iloc[1:].reset_index(drop=True)
        
        mini_dfs.append(final_mini_df)

    # Concatenate all mini-dataframes into one
    result_df_part1 = pd.concat(mini_dfs, ignore_index=True)

    df_2 = df_2.drop(df_2.index[5:][df_2.iloc[5:, 1].isna() | (df_2.iloc[5:, 1].str.contains("SUB-TOTAL|TOTAL", na=False))])
    df_2 = df_2.drop(df_2.index[:1]).reset_index(drop=True)
    df_2.iloc[1] = df_2.iloc[1].ffill()
    # Filter condition to keep columns where row 2 contains both "CAMAS" and ("FEBRILES" or "UVICLIN") (case insensitive)
    filter_condition_2 = df_2.iloc[1].apply(lambda x: 'camas' in str(x).lower() and ('febriles' in str(x).lower() or 'uviclin' in str(x).lower()))
    selected_columns_2 = df_2.columns[filter_condition_2]

    # Create a new DataFrame with the first 4 columns and the selected columns
    df_2_filtered = df_2.loc[:, df_2.columns[:4].tolist() + selected_columns_2.tolist()]
    
    def setup_and_process_dataframe(df_2_filtered):
        if not df_2_filtered.empty:
            # Fill NaNs in the first four columns downwards from the first row
            df_2_filtered.iloc[:, :4] = df_2_filtered.iloc[:, :4].ffill()

            # Fill NaNs in columns 5 and onward both upwards and downwards from row 1
            for col in df_2_filtered.columns[4:]:
                df_2_filtered[col] = df_2_filtered[col].ffill().bfill()

            # Extract column names from the fourth row (index 3)
            new_columns = list(df_2_filtered.iloc[3])

            # Set new columns for the DataFrame
            df_2_filtered.columns = new_columns

            # Drop row 3 (index 3) and reset the index
            df_2_filtered = df_2_filtered.drop(3).reset_index(drop=True)

            # Process mini DataFrames
            base_columns = df_2_filtered.iloc[3:, :4].reset_index(drop=True)  # Selects the first four columns as base columns, excluding the first three rows
            num_pairs = (df_2_filtered.shape[1] - 4) // 2  # Calculates the number of pairs of columns

            mini_dfs = []  # Initializes an empty list to store mini DataFrames

            # Loop through each pair and create a mini-dataframe
            for i in range(num_pairs):
                total_col = 4 + 2 * i
                ocup_col = total_col + 1

                # Create a mini DataFrame for the current pair
                mini_df = pd.concat([base_columns, df_2_filtered.iloc[:, [total_col, ocup_col]]], axis=1)

                # Extract the first three rows for melting
                tip_cama_1 = mini_df.iloc[0, 4:].reset_index(drop=True)
                tip_cama_2 = mini_df.iloc[1, 4:].reset_index(drop=True)
                tip_cama_3 = mini_df.iloc[2, 4:].reset_index(drop=True)

                # Create a DataFrame from the melted rows
                melted_df = pd.DataFrame({
                    'TIP_CAMA_1': tip_cama_1,
                    'TIP_CAMA_2': tip_cama_2,
                    'TIP_CAMA_3': tip_cama_3
                })

                # Fill NaNs in the melted_df
                melted_df = melted_df.ffill()

                # Drop the first three rows from the mini DataFrame
                remaining_df = mini_df.drop(index=range(3)).reset_index(drop=True)

                # Concatenate mini DataFrame with melted values in the correct order
                final_mini_df = pd.concat([base_columns.iloc[:remaining_df.shape[0]].reset_index(drop=True),
                                        melted_df, remaining_df.iloc[:, 4:].reset_index(drop=True)], axis=1)

                # Fill the TIP_CAMA columns downwards in the whole column
                final_mini_df[['TIP_CAMA_1', 'TIP_CAMA_2', 'TIP_CAMA_3']] = final_mini_df[['TIP_CAMA_1', 'TIP_CAMA_2', 'TIP_CAMA_3']].ffill()

                # Append final mini DataFrame to list
                mini_dfs.append(final_mini_df)

            # Concatenate all mini-dataframes into one
            result_df = pd.concat(mini_dfs, ignore_index=True)

            return result_df

    # Process the second part dataframe
    result_df_part2 = setup_and_process_dataframe(df_2_filtered)

    # Concatenate the results from both parts
    result_df = pd.concat([result_df_part1, result_df_part2], ignore_index=True)

    # Adding a new column which is the subtraction of 'Ocup' from 'Total'
    result_df['Disp'] = result_df['Total'] - result_df['Ocup']

    # Create a new column at the end with the value from the first column and row
    result_df['PERIODO'] = value

    return result_df

# Initialize an empty DataFrame to store all processed data
Camas_df = pd.DataFrame()

# Process each file, append the results to the final DataFrame incrementally
for file_path in file_list:
    processed_df = process_file(file_path)
    Camas_df = pd.concat([Camas_df, processed_df], ignore_index=True)

# Now `Camas_df` contains all the processed data and is ready to be uploaded to a database
with engine2.begin() as connection:
    for date in recent_dates_timestamps:
        query = text("""DELETE FROM camas_essalud WHERE "PERIODO"::date = :date""")
        connection.execute(query, {"date": date})
Camas_df.to_sql(name=f'camas_essalud', con=engine2, if_exists='append', index=False,chunksize=500000)

In [19]:
# DATAFRAME CONSULTORIOS FEBRILES

# Function to process each file for CONSULTORIOS FEBRILES
def process_file_febriles(file_path):
    df = pd.read_excel(file_path, sheet_name='IPRESS')
    df_3 = df.copy()
    value_periodo = df_3.columns[0]
    
    # Drop rows where the condition is met
    df_3 = df_3.drop(df_3.index[5:][df_3.iloc[5:, 1].isna() | (df_3.iloc[5:, 1].str.contains("SUB-TOTAL|TOTAL", na=False))])
    
    # Drop first row and reset index
    df_3 = df_3.drop(df_3.index[:1]).reset_index(drop=True)
    
    # Forward fill NaN values in the second row
    df_3.iloc[1] = df_3.iloc[1].ffill()
    
    # Filter columns where row 2 contains both "CONSULTORIOS" and "FEBRILES" (case insensitive)
    filter_condition_3 = df_3.iloc[1].apply(lambda x: 'consultorios' in str(x).lower() and 'febriles' in str(x).lower())
    selected_columns_3 = df_3.columns[filter_condition_3]
    
    # Create a new DataFrame with the first 4 columns and the selected columns
    df_3_filtered = df_3.loc[:, df_3.columns[:4].tolist() + selected_columns_3.tolist()]
    
    def setup_and_process_dataframe(df_3_filtered):
        if not df_3_filtered.empty:
            # Fill NaNs in the first four columns downwards from the first row
            df_3_filtered.iloc[:, :4] = df_3_filtered.iloc[:, :4].ffill()

            # Fill NaNs in columns 5 and onward both upwards and downwards from row 1
            for col in df_3_filtered.columns[4:]:
                df_3_filtered[col] = df_3_filtered[col].ffill().bfill()

            # Extract column names from the fourth row (index 3)
            new_columns = list(df_3_filtered.iloc[3])

            # Set new columns for the DataFrame
            df_3_filtered.columns = new_columns

            # Drop row 3 (index 3) and reset the index
            result_df_3 = df_3_filtered.drop(3).reset_index(drop=True)

            # Dropping 3 rows and reindexing
            result_df_3 = result_df_3.drop(range(3)).reset_index(drop=True)

            # Create Periodo Column
            result_df_3['PERIODO'] = value_periodo

            return result_df_3

    # Process the DataFrame
    result_df_3 = setup_and_process_dataframe(df_3_filtered)

    return result_df_3

# Initialize an empty DataFrame to store all processed data
Consultorios_Febriles_df = pd.DataFrame()

# Process each file, append the results to the final DataFrame incrementally
for file_path in file_list:
    processed_df = process_file_febriles(file_path)
    Consultorios_Febriles_df = pd.concat([Consultorios_Febriles_df, processed_df], ignore_index=True)

# Now `Consultorios_Febriles_df` contains all the processed data and is ready to be used
with engine2.begin() as connection:
    for date in recent_dates_timestamps:
        query = text("""DELETE FROM unidades_febriles WHERE "PERIODO"::date = :date""")
        connection.execute(query, {"date": date})
Consultorios_Febriles_df.to_sql(name=f'unidades_febriles', con=engine2, if_exists='append', index=False,chunksize=500000)


Consultorios Febriles DataFrame:
     RED ASISTENCIAL                                    IPRESS Conformidad?             Registro Nro de Consultorios de Febriles             PERIODO
0           ALMENARA                          H I A DIAZ UFANO           Si  2024-04-26 07:19:00                               0 2024-04-26 09:01:00
1           ALMENARA                         H I VOTO BERNALES           Si  2024-04-26 07:18:00                               0 2024-04-26 09:01:00
2           ALMENARA                             H II CASTILLA           Si  2024-04-26 08:22:00                               0 2024-04-26 09:01:00
3           ALMENARA                    H II S ISIDRO LABRADOR           Si  2024-04-26 07:31:00                               0 2024-04-26 09:01:00
4           ALMENARA                              H II VITARTE           Si  2024-04-26 07:22:00                               0 2024-04-26 09:01:00
5           ALMENARA                    H III EMERGENCIAS GRAU          

324